# Pydantic model to structure output

create an agent to simulate IT employees

In [1]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent("openrouter:liquid/lfm-2.5-1.2b-thinking:free",
system_prompt="""
You are a HR expert within IT field in Sweden witihin data science, data enginerring, 
machine larning, AI engineering. You will simulate IT employees. 

Fields to include in output:
- Name
- age
- gender
- job_title
- salary in SEK per month
""")

result = await employee_simulator_agent.run("Simulate two employees")

In [2]:
print(result.output)

Here are two simulated IT employee profiles based on typical roles and data fields:

1. **Employee 1**  
   - **Name:** Erik  
   - **Age:** 32  
   - **Gender:** Male  
   - **Job Title:** Senior Developer  
   - **Salary:** 380,000 SEK/month  

2. **Employee 2**  
   - **Name:** Anika  
   - **Age:** 29  
   - **Gender:** Female  
   - **Job Title:** Data Analyst  
   - **Salary:** 240,000 SEK/month  

### Key Observations:  
- **Salary Rationale:** Aligned with global market rates for roles in Sweden (e.g., Senior Developer reflects tech-tier compensation; Data Analyst closer to entry-level).  
- **Data Sources:** Modeled from HR records, salary surveys, and industry benchmarks (ECTE, SAP Research).  
- **Format:** Structured for ease of performance evaluation metrics (e.g., age-weighted compensation adjustments).  

Let me know if you need adjustments or additional details!


In [3]:
with open("simulated_employees.md", "w") as file:
    file.write(result.output)

## Get more structured output

Issue with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repeatable structure

In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent


class EmployeeModel(BaseModel):
    name: str = Field(
        description="Mostly swedish names, but could be foreign names as well"
    )
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000,
        lte=50_000,
        description="salary should be between 30k and 50k, the higher experience level, the higher salary",
    )


employee_simulator_agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""",
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=EmployeeModel)
result

/var/folders/ls/2byy4k1d75jcd5jnq7blvr4c0000gn/T/ipykernel_30248/1484438454.py:14: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


AgentRunResult(output=EmployeeModel(name='Oskar Lindberg', age=29, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=44000))

In [6]:
result.output

EmployeeModel(name='Oskar Lindberg', age=29, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=44000)

In [7]:
result.output.salary+5000

49000

In [8]:
employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt=""" 
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel], )
result

AgentRunResult(output=[EmployeeModel(name='Anna', age=30, gender='Female', experience_level='Senior', job_title='Senior Software Developer', salary=45000), EmployeeModel(name='Erik', age=28, gender='Male', experience_level='Mid level', job_title='Data Scientist', salary=38000), EmployeeModel(name='Sofia', age=22, gender='Female', experience_level='Entry', job_title='Machine Learning Engineer', salary=32000)])

In [9]:
result.output[0]

EmployeeModel(name='Anna', age=30, gender='Female', experience_level='Senior', job_title='Senior Software Developer', salary=45000)

In [10]:
# Basemodel -> dictionary
result.output[0].model_dump()

{'name': 'Anna',
 'age': 30,
 'gender': 'Female',
 'experience_level': 'Senior',
 'job_title': 'Senior Software Developer',
 'salary': 45000}

TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on this list
- export a csv file of our simulated employees

In [14]:
employees = result.output
employees

[EmployeeModel(name='Anna', age=30, gender='Female', experience_level='Senior', job_title='Senior Software Developer', salary=45000),
 EmployeeModel(name='Erik', age=28, gender='Male', experience_level='Mid level', job_title='Data Scientist', salary=38000),
 EmployeeModel(name='Sofia', age=22, gender='Female', experience_level='Entry', job_title='Machine Learning Engineer', salary=32000)]

In [15]:
employees_list = [employee.model_dump() for employee in employees]
employees_list

[{'name': 'Anna',
  'age': 30,
  'gender': 'Female',
  'experience_level': 'Senior',
  'job_title': 'Senior Software Developer',
  'salary': 45000},
 {'name': 'Erik',
  'age': 28,
  'gender': 'Male',
  'experience_level': 'Mid level',
  'job_title': 'Data Scientist',
  'salary': 38000},
 {'name': 'Sofia',
  'age': 22,
  'gender': 'Female',
  'experience_level': 'Entry',
  'job_title': 'Machine Learning Engineer',
  'salary': 32000}]

In [17]:
import pandas as pd 

df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Anna,30,Female,Senior,Senior Software Developer,45000
1,Erik,28,Male,Mid level,Data Scientist,38000
2,Sofia,22,Female,Entry,Machine Learning Engineer,32000


In [18]:
df["salary"].mean()

np.float64(38333.333333333336)

In [19]:
df.to_csv("simulated_employees.csv", index=False)